In [1]:
from manim import *
from scipy.stats import norm
import numpy as np

In [4]:
from __future__ import annotations
from manim import *
import numpy as np

class TensionSpring(VMobject):
    """
    A procedural spring that connects two points.
    Designed to be used with always_redraw for dynamic animations.
    Uses self.start_point and self.end_point to avoid attribute conflicts with VMobject internals.
    """
    def __init__(self, start: np.ndarray, end: np.ndarray, 
                 coils: int = 8, radius: float = 0.1, color = RED, **kwargs):
        super().__init__(color=color, **kwargs)
        self.start_point = np.array(start)
        self.end_point = np.array(end)
        self.coils = coils
        self.radius = radius
        self.generate_points()

    def generate_points(self):
        vec = self.end_point - self.start_point
        length = np.linalg.norm(vec)
        
        # Handle edge case where start == end
        if length < 1e-6:
            self.set_points_as_corners([self.start_point, self.end_point])
            return

        direction = vec / length
        # Vector perpendicular to direction (for the coil width)
        perp = np.array([-direction[1], direction[0], 0])
        
        # Create zigzag pattern
        t_range = np.linspace(0, 1, self.coils * 12 + 1)
        points = []
        for t in t_range:
            sine_wave = self.radius * np.sin(2 * np.pi * self.coils * t) * perp
            linear_progress = self.start_point + (direction * length * t)
            points.append(linear_progress + sine_wave)
            
        self.set_points_as_corners(points)

# --- 2. Helper for Dynamic Redrawing ---
def get_dynamic_spring(mobj1: Mobject, mobj2: Mobject, color=RED) -> Mobject:
    """Returns a spring that automatically updates when mobj1 or mobj2 moves."""
    return always_redraw(lambda: TensionSpring(
        start=mobj1.get_top() + DOWN*0.05, # Anchor slightly inside
        end=mobj2.get_top() + DOWN*0.05,
        coils=10,
        radius=0.08,
        color=color,
        stroke_width=3
    ))

In [5]:
%%manim -v WARNING --fps 30 -ql RankingLossScene
    def construct(self):
        # --- Data Setup ---
        # 5 items. Index 2 (Blue) and Index 3 (Orange) are the conflict.
        # Blue (Rel=1) has score 2.5 (Too Low)
        # Orange (Rel=0) has score 3.5 (Too High)
        scores = np.array([2.0, 1.5, 2.5, 3.5, 1.0]) 
        relevance = np.array([1, 1, 1, 0, 0]) 
        
        # --- Visual Setup ---
        bar_width = 0.6
        spacing = 1.0
        
        bars = VGroup()
        for i, score in enumerate(scores):
            # Color logic: Relevant = Blue, Irrelevant = Orange
            color = BLUE if relevance[i] == 1 else ORANGE
            
            bar = Rectangle(
                width=bar_width, height=score, 
                fill_color=color, fill_opacity=0.8, stroke_color=WHITE, stroke_width=1
            )
            # Position logic
            x = (i - 2) * spacing
            bar.move_to([x, -2, 0], aligned_edge=DOWN)
            
            # Label
            lbl = Text(f"s_{i}", font_size=20, color=LIGHT_GREY).next_to(bar, DOWN)
            val = DecimalNumber(score, num_decimal_places=1, font_size=20).next_to(bar, UP)
            
            bars.add(VGroup(bar, lbl, val))
            
        # Ground line
        ground = Line(LEFT*4, RIGHT*4, color=GREY).move_to(DOWN*2)
        
        # Formula with color coding
        formula = MathTex(
            r"\mathcal{L}_{rank} = \sum \max(0, 1 - (",
            r"s_{pos}", 
            r"-", 
            r"s_{neg}",
            r"))"
        ).to_edge(UP)
        formula[1].set_color(BLUE)
        formula[3].set_color(ORANGE)
        
        self.add(ground)
        self.play(
            LaggedStart(*[GrowFromEdge(b[0], DOWN) for b in bars], lag_ratio=0.1),
            Write(formula)
        )
        self.play(FadeIn(VGroup(*[b[1:] for b in bars]))) # Fade in labels
        self.wait(0.5)

        # --- Identify Conflict ---
        idx_pos, idx_neg = 2, 3
        group_pos = bars[idx_pos] # The Blue one
        group_neg = bars[idx_neg] # The Orange one
        
        # Dim others
        others = VGroup(*[b for i, b in enumerate(bars) if i not in [idx_pos, idx_neg]])
        self.play(others.animate.set_opacity(0.3))
        
        # Highlight pair
        self.play(
            group_pos.animate.scale(1.1),
            group_neg.animate.scale(1.1),
            Indicate(group_pos[0], color=BLUE_A),
            Indicate(group_neg[0], color=ORANGE),
        )

        # --- Physics Animation ---
        
        # 1. Attach Dynamic Spring
        # Note: We attach to the bar rect (index 0 of the group)
        # Defensive: Make sure group_pos[0] and group_neg[0] have .get_top()
        # Remove any code referencing a missing '.end' attribute.
        spring = get_dynamic_spring(group_pos[0], group_neg[0])
        
        violation_text = Text("Violation!", font_size=24, color=RED).next_to(spring, UP, buff=0.5)
        
        self.play(Create(spring))
        self.play(Write(violation_text))
        self.play(spring[0].animate.set_stroke(width=5), rate_func=wiggle)
        
        # 2. Animate the Swap (Spring stretches/shrinks automatically)
        pos_x = group_pos.get_x()
        neg_x = group_neg.get_x()
        
        # Arc path to look like they are jumping over each other
        self.play(
            group_pos.animate.set_x(neg_x),
            group_neg.animate.set_x(pos_x),
            FadeOut(violation_text),
            run_time=1.5,
            rate_func=smooth
        )
        
        # 3. Snap Spring Removal
        self.play(FadeOut(spring), scale=0.5)
        
        # 4. Resolve & Restore
        check = Text("Fixed!", color=GREEN, font_size=24).next_to(group_pos, UP, buff=0.5)
        self.play(
            others.animate.set_opacity(1),
            group_pos.animate.scale(1/1.1),
            group_neg.animate.scale(1/1.1),
            Transform(violation_text, check)
        )
        self.wait(2)

IndentationError: unexpected indent (<string>, line 1)

In [4]:
%%manim -v WARNING --fps 30 -ql UncertaintyLossScene

class UncertaintyLossScene(Scene):
    def construct(self):
        # 1. Setup Axes
        axes = Axes(
            x_range=[-2, 6, 1], y_range=[0, 1.5, 0.5],
            axis_config={"include_tip": False}
        ).scale(0.8).shift(DOWN*0.5)
        
        # 2. Trackers for animation
        mu_t = ValueTracker(0.0) # Bad prediction (far left)
        sigma_t = ValueTracker(1.2) # High uncertainty
        gt_x = 3.5
        
        # 3. Mobjects
        cloud_grp = create_gaussian_plot(axes, mu_t, sigma_t)
        
        gt_line = DashedLine(
            axes.c2p(gt_x, 0), axes.c2p(gt_x, 1.5), 
            color=YELLOW, stroke_width=4
        )
        gt_lbl = Text("Ground Truth", font_size=20, color=YELLOW).next_to(gt_line, UP)
        
        title = Title("Uncertainty Loss (NLL)").scale(0.8)
        
        self.add(title, axes, gt_line, gt_lbl)
        self.play(FadeIn(cloud_grp))
        self.wait(1)
        
        # 4. Phase 1: Shift Mean (Accuracy)
        arrow = Arrow(axes.c2p(0, 0.5), axes.c2p(3, 0.5), color=RED)
        txt = Text("Minimize Error", font_size=24, color=RED).next_to(arrow, UP)
        
        self.play(GrowArrow(arrow), Write(txt))
        self.play(
            mu_t.animate.set_value(gt_x),
            FadeOut(arrow), FadeOut(txt),
            run_time=2,
            rate_func=smooth
        )
        
        # 5. Phase 2: Shrink Variance (Precision)
        # Note: We clamp scale to avoid division by zero artifacts
        arrows = VGroup(
            Arrow(axes.c2p(gt_x - 1.5, 0.3), axes.c2p(gt_x - 0.5, 0.3), color=PURPLE),
            Arrow(axes.c2p(gt_x + 1.5, 0.3), axes.c2p(gt_x + 0.5, 0.3), color=PURPLE)
        )
        txt_2 = Text("Minimize Variance", font_size=24, color=PURPLE).next_to(cloud_grp, LEFT)
        
        self.play(FadeIn(arrows), Write(txt_2))
        self.play(
            sigma_t.animate.set_value(0.4),
            FadeOut(arrows), FadeOut(txt_2),
            run_time=2
        )
        
        final_txt = Text("High Confidence Match", font_size=32, color=GREEN).to_edge(DOWN)
        self.play(Write(final_txt))
        self.wait(2)

Manim Community v0.19.1

NameError: name 'create_gaussian_plot' is not defined

In [5]:
%%manim -v WARNING --fps 30 -ql CombinedLossScene

class CombinedLossScene(Scene):
    """
    Demonstrates that Ranking and Uncertainty run in parallel.
    Uses the helper functions to build the view without nesting Scenes.
    """
    def construct(self):
        # Layout Division
        self.play(Write(Text("Total Loss = Ranking + Uncertainty", font_size=36).to_edge(UP)))
        
        left_zone = VGroup().to_edge(LEFT, buff=1)
        right_zone = VGroup().to_edge(RIGHT, buff=1)
        
        # --- LEFT: Ranking Setup ---
        scores = [2, 4, 3] 
        relevance = [1, 0, 0] # Item 0 is relevant but score is low
        r_layout, r_bars = create_ranking_mobjects(scores, relevance)
        r_layout.scale(0.7).move_to(LEFT * 3.5 + DOWN)
        r_title = Text("Ranking", font_size=24, color=ORANGE).next_to(r_layout, UP)
        
        # --- RIGHT: Uncertainty Setup ---
        axes = Axes(x_range=[0, 5, 1], y_range=[0, 2, 1], x_length=4, y_length=3).scale(0.8)
        axes.move_to(RIGHT * 3.5 + DOWN)
        mu_t = ValueTracker(1.0)
        sigma_t = ValueTracker(0.8)
        gt_x = 3.0
        
        u_plot = create_gaussian_plot(axes, mu_t, sigma_t, color=BLUE)
        u_gt = Dot(axes.c2p(gt_x, 0), color=YELLOW, radius=0.1)
        u_title = Text("Uncertainty", font_size=24, color=BLUE).next_to(axes, UP)
        
        # Add Initial State
        self.play(
            FadeIn(r_layout), Write(r_title),
            FadeIn(axes), FadeIn(u_plot), FadeIn(u_gt), Write(u_title)
        )
        
        # --- ANIMATE BOTH SIMULTANEOUSLY ---
        
        # 1. Prepare Ranking Animation (Swap bars 0 and 1)
        bar_relevant = r_bars[0]
        bar_irrelevant = r_bars[1]
        
        # Create Spring
        spring = TensionSpring(bar_relevant[0].get_top(), bar_irrelevant[0].get_top(), color=RED)
        
        # 2. Execute Parallel Optimization
        self.play(Create(spring), run_time=1)
        
        self.play(
            # Ranking Action: Swap positions
            bar_relevant.animate.set_x(bar_irrelevant.get_x()),
            bar_irrelevant.animate.set_x(bar_relevant.get_x()),
            FadeOut(spring),
            
            # Uncertainty Action: Move Mu to GT and Shrink Sigma
            mu_t.animate.set_value(gt_x),
            sigma_t.animate.set_value(0.3),
            
            run_time=3,
            rate_func=smooth
        )
        
        self.wait(2)

Manim Community v0.19.1

NameError: name 'create_ranking_mobjects' is not defined

In [2]:
%%manim -v WARNING --fps 120 --disable_caching -qk HybridArchitecture

COLOR_LSTM = "#9370DB"
COLOR_EMBED = "#FFA500"
COLOR_FUSION = "#90EE90"
COLOR_ATTN = "#00BFFF"
COLOR_MU = BLUE
COLOR_VAR = RED
COLOR_TARGET = GREEN
COLOR_RANK = ORANGE
COLOR_LOSS = RED

class HybridArchitecture(ThreeDScene):
    def construct(self):
        self.play_input_cube()
        self.play_lstm_path()
        self.play_identity_path()
        self.play_fusion()
        self.play_transformer_network()

    def play_input_cube(self):
        self.set_camera_orientation(phi=75 * DEGREES, theta=-45 * DEGREES)
        depth_stock = 4.0
        width_time = 4.0
        height_feat = 2.0

        bulk_tensor = Prism(
            dimensions=[width_time, height_feat, depth_stock - 0.2],
            fill_color=BLUE,
            fill_opacity=0.2,
            stroke_color=BLUE_E,
            stroke_width=1
        ).move_to(IN * 0.2)

        slice_bg = Prism(
            dimensions=[width_time, height_feat, 0.2],
            fill_color=BLUE_E,
            fill_opacity=0.5,
            stroke_color=WHITE,
            stroke_width=2
        ).move_to(OUT * (depth_stock/2))

        grid_lines = VGroup()
        for x in np.linspace(-width_time/2, width_time/2, 10):
            l = Line(
                start=[x, -height_feat/2, depth_stock/2 + 0.1], 
                end=[x, height_feat/2, depth_stock/2 + 0.1],
                color=WHITE, stroke_width=1, stroke_opacity=0.5
            )
            grid_lines.add(l)
        for y in np.linspace(-height_feat/2, height_feat/2, 4):
            l = Line(
                start=[-width_time/2, y, depth_stock/2 + 0.1], 
                end=[width_time/2, y, depth_stock/2 + 0.1],
                color=WHITE, stroke_width=1, stroke_opacity=0.5
            )
            grid_lines.add(l)
        
        self.hero_group = VGroup(slice_bg, grid_lines)
        tensor_group = VGroup(bulk_tensor, self.hero_group)
        tensor_group.scale(0.7).shift(DOWN * 0.5)

        lbl_stocks = Text("N=53 Stocks", font_size=24).rotate(PI/2, axis=RIGHT).next_to(bulk_tensor, UP)
        lbl_time = Text("T=60 Days", font_size=24).rotate(PI/2, axis=RIGHT).next_to(self.hero_group, DOWN)
        lbl_feats = Text("F=6 Features", font_size=24).rotate(PI/2, axis=RIGHT).next_to(self.hero_group, RIGHT)
        labels_3d = VGroup(lbl_stocks, lbl_time, lbl_feats)

        self.play(DrawBorderThenFill(tensor_group), Write(labels_3d))
        self.begin_ambient_camera_rotation(rate=0.2)
        self.wait(2.5)
        self.stop_ambient_camera_rotation()
        self.move_camera(phi=60 * DEGREES, theta=-45 * DEGREES, run_time=1)
        self.play(
            self.hero_group.animate.shift(OUT * 0.5 + LEFT * 1.5),
            bulk_tensor.animate.set_opacity(0.05),
            FadeOut(labels_3d),
            run_time=2
        )
        slice_label = Text("Single Stock History", font_size=24, color=YELLOW).rotate(PI/2, axis=RIGHT).next_to(self.hero_group, UP)
        self.play(Write(slice_label), grid_lines.animate.set_color(YELLOW))
        self.wait(0.5)
        self.play(
            FadeOut(bulk_tensor),
            slice_label.animate.rotate(-90*DEGREES, axis=RIGHT).move_to(LEFT * 4 + UP * 1).scale(0.8),            
            FadeOut(slice_label),
            self.hero_group.animate.rotate(-90*DEGREES, axis=RIGHT).move_to(LEFT * 4 + UP * 1).scale(0.8),
            run_time=2
        )
        self.move_camera(phi=0, theta=-90*DEGREES, run_time=1.5)

    def play_lstm_path(self):
        import random

        num_visible_steps = 6
        inputs = VGroup()
        for i in range(num_visible_steps):
            dot = Circle(radius=0.25, color=BLUE, fill_opacity=0.5, stroke_width=2)
            label = MathTex(f"x_{{{i+1}}}", font_size=24).move_to(dot)
            group = VGroup(dot, label)
            inputs.add(group)
        inputs.arrange(RIGHT, buff=1.2).shift(DOWN * 2)
        self.play(
            LaggedStart(*[GrowFromPoint(inp, self.hero_group.get_center()) for inp in inputs], lag_ratio=0.1),
            run_time=1.5
        )
        self.play(FadeOut(self.hero_group))

        lstm_cells = VGroup()
        arrows_recurrence = VGroup()
        arrows_input = VGroup()
        cell_height = 1.2
        for i in range(num_visible_steps):
            cell = RoundedRectangle(corner_radius=0.2, height=cell_height, width=1.2, color=COLOR_LSTM, fill_opacity=0.2)
            cell.move_to(inputs[i].get_center() + UP * 2)
            lstm_cells.add(cell)
            arr_in = Arrow(inputs[i].get_top(), cell.get_bottom(), buff=0.1, color=BLUE, max_tip_length_to_length_ratio=0.15)
            arrows_input.add(arr_in)
            if i > 0:
                arr_rec = Arrow(
                    lstm_cells[i-1].get_right(), 
                    cell.get_left(), 
                    buff=0.1, 
                    color=COLOR_LSTM,
                    max_tip_length_to_length_ratio=0.15
                )
                arrows_recurrence.add(arr_rec)

        eq_lstm = MathTex(
            r"h_t = \text{LSTM}(h_{t-1}, x_t)", 
            font_size=36, 
            color=COLOR_LSTM
        ).to_edge(UP)
        self.play(Write(eq_lstm))
        for i in range(num_visible_steps):
            anims = []
            anims.append(Create(arrows_input[i]))
            packet_in = Dot(radius=0.08, color=YELLOW).move_to(inputs[i].get_center())
            anims.append(packet_in.animate.move_to(lstm_cells[i].get_center()))
            if i > 0:
                anims.append(Create(arrows_recurrence[i-1]))
                packet_rec = Dot(radius=0.08, color=YELLOW).move_to(lstm_cells[i-1].get_center())
                anims.append(packet_rec.animate.move_to(lstm_cells[i].get_center()))
            anims.append(GrowFromCenter(lstm_cells[i]))
            self.play(AnimationGroup(*anims), run_time=0.5)
            self.remove(packet_in)
            if i > 0: self.remove(packet_rec)

        dropout_dots = VGroup()
        for cell in lstm_cells:
            neurons = VGroup(*[Dot(radius=0.035, color=WHITE) for _ in range(49)])
            neurons.arrange_in_grid(rows=7, cols=7, buff=0.07).move_to(cell)
            dropout_dots.add(neurons)
        self.play(FadeIn(dropout_dots))
        dropout_label = Text("Dropout 44%", font_size=20).next_to(lstm_cells, UP)
        self.play(FadeIn(dropout_label))
        self.wait(0.5)
        fizz_anims = []
        for neurons in dropout_dots:
            for neuron in neurons:
                if random.random() < 0.44:
                    fizz_anims.append(
                        AnimationGroup(
                            neuron.animate.set_color(RED).scale(1.5),
                            neuron.animate.set_opacity(0).scale(0),
                            lag_ratio=0.1
                        )
                    )
        self.play(AnimationGroup(*fizz_anims, lag_ratio=0.05), run_time=2)
        self.play(FadeOut(dropout_label))

        last_cell = lstm_cells[-1]
        output_vec = VGroup(
            RoundedRectangle(height=2, width=0.4, corner_radius=0.1, color=COLOR_EMBED, fill_opacity=0.8),
            MathTex(r"h_{60}", color=WHITE, font_size=24)
        )
        output_vec[1].move_to(output_vec[0])
        output_vec.next_to(last_cell, RIGHT, buff=1.5)
        output_label = MathTex(r"h_{60} \in \mathbb{R}^{128}", font_size=24).next_to(output_vec, UP)
        arrow_out = Arrow(last_cell.get_right(), output_vec.get_left(), color=COLOR_LSTM)
        packet_out = Dot(color=YELLOW).move_to(last_cell.get_center())
        self.play(Create(arrow_out))
        self.play(
            packet_out.animate.move_to(output_vec.get_center()),
            GrowFromCenter(output_vec),
            Write(output_label)
        )
        self.play(FadeOut(packet_out))
        self.lstm_vec = output_vec[0]
        self.vec_label_t = output_vec[1]
        self.lbl_momentum = output_label

        cleanup_group = VGroup(
            inputs, lstm_cells, arrows_input, arrows_recurrence, 
            dropout_dots, eq_lstm, arrow_out
        )
        self.play(FadeOut(cleanup_group))

    def play_identity_path(self):
        COLOR_PACKET = "#FFFF00"
        start_pos = LEFT * 3.5 + DOWN * 2.0

        chip_group = VGroup(
            Circle(radius=0.3, color=COLOR_EMBED, fill_opacity=0.2, stroke_width=2),
            Text("ID", font_size=16, color=COLOR_EMBED)
        ).move_to(start_pos)
        chip_label = Text("Stock Identity", font_size=25, color=COLOR_EMBED).next_to(chip_group, UP)
        self.play(FadeIn(chip_group), Write(chip_label))

        rows = 4
        matrix_group = VGroup()
        for i in range(rows):
            rect = Rectangle(height=0.25, width=1.5, color=BLUE_E, fill_opacity=0.5, stroke_width=1)
            matrix_group.add(rect)
        matrix_group.arrange(DOWN, buff=0).next_to(chip_group, RIGHT, buff=1.5)

        matrix_label = Text("Embedding Layer", font_size=16, color=BLUE).next_to(matrix_group, UP)
        arrow_in = Arrow(chip_group.get_right(), matrix_group.get_left(), color=COLOR_EMBED, buff=0.1)
        self.play(Create(matrix_group), FadeIn(matrix_label), GrowArrow(arrow_in))

        packet = Dot(radius=0.06, color=COLOR_PACKET).move_to(chip_group.get_center())
        self.play(MoveAlongPath(packet, Line(chip_group.get_center(), matrix_group.get_center())), run_time=0.6)

        target_row = matrix_group[1]
        self.play(
            FadeOut(packet),
            Indicate(target_row, color=COLOR_EMBED, scale_factor=1.2),
            run_time=0.5
        )

        self.id_vec = RoundedRectangle(height=0.4, width=1.2, corner_radius=0.1, fill_color=COLOR_EMBED, fill_opacity=0.9, stroke_width=0)
        self.id_vec.next_to(matrix_group, RIGHT, buff=1.0)
        self.vec_label_id = Text("Emb Vector", font_size=14, color=BLACK).move_to(self.id_vec)
        arrow_out = Arrow(matrix_group.get_right(), self.id_vec.get_left(), color=COLOR_EMBED)

        self.play(GrowArrow(arrow_out))
        self.play(
            TransformFromCopy(target_row, self.id_vec),
            Write(self.vec_label_id)
        )

        self.lbl_static = Text("Static Traits (Sector)", font_size=25, color=WHITE).next_to(self.id_vec, DOWN)
        self.play(FadeIn(self.lbl_static))

        self.play(
            FadeOut(chip_group), FadeOut(chip_label),
            FadeOut(matrix_group), FadeOut(matrix_label),
            FadeOut(arrow_in), FadeOut(arrow_out)
        )

    def play_fusion(self):
        center_point = ORIGIN

        self.play(
            self.lstm_vec.animate.next_to(center_point, LEFT, buff=0.1),
            self.vec_label_t.animate.next_to(center_point, LEFT, buff=0.1).shift(RIGHT*0.1),
            FadeOut(self.lbl_momentum),
            self.id_vec.animate.next_to(center_point, RIGHT, buff=0.1),
            self.vec_label_id.animate.next_to(center_point, RIGHT, buff=0.1).shift(LEFT*0.05),
            FadeOut(self.lbl_static)
        )

        brace = Brace(VGroup(self.lstm_vec, self.id_vec), UP, color=WHITE)
        concat_text = brace.get_text("Concatenate").scale(0.8)
        self.play(GrowFromCenter(brace), FadeIn(concat_text))
        self.wait(0.5)

        self.fused_vec = RoundedRectangle(height=0.5, width=4, corner_radius=0.1, fill_color=COLOR_FUSION, fill_opacity=1, stroke_width=0)
        self.fused_vec.move_to(center_point)
        self.fused_label = Text("Fused State (144 dims)", font_size=24, color=BLACK).move_to(self.fused_vec)
        self.play(
            ReplacementTransform(VGroup(self.lstm_vec, self.id_vec), self.fused_vec),
            ReplacementTransform(VGroup(self.vec_label_t, self.vec_label_id), self.fused_label),
            FadeOut(brace), FadeOut(concat_text)
        )
        self.play(Indicate(self.fused_vec, color=WHITE, scale_factor=1.05))
        self.wait(0.5)

    def play_transformer_network(self):
        target_phi = 70 * DEGREES
        target_theta = -30 * DEGREES
        self.move_camera(phi=target_phi, theta=target_theta, run_time=1.5)
        
        num_nodes = 64
        radius = 3.0
        nodes = VGroup()
        
        golden_ratio = (1 + 5 ** 0.5) / 2
        node_positions = []

        for i in range(num_nodes):
            theta = 2 * PI * i / golden_ratio
            phi = np.arccos(1 - 2 * (i + 0.5) / num_nodes)
            x = np.cos(theta) * np.sin(phi) * radius
            y = np.sin(theta) * np.sin(phi) * radius
            z = np.cos(phi) * radius
            pos = np.array([x, y, z])
            node_positions.append(pos)
            dot = Dot3D(point=pos, radius=0.08, color=COLOR_FUSION)
            nodes.add(dot)

        self.begin_ambient_camera_rotation(rate=0.1)
        self.play(
            ReplacementTransform(VGroup(self.fused_vec, self.fused_label), nodes),
            run_time=2
        )

        lines = VGroup()
        for i in range(num_nodes):
            targets = np.random.choice(range(num_nodes), 5, replace=False)
            for t in targets:
                if i != t:
                    dist = np.linalg.norm(node_positions[i] - node_positions[t])
                    opacity = max(0.05, 0.4 - (dist * 0.05))
                    l = Line(
                        node_positions[i], 
                        node_positions[t], 
                        stroke_width=1.5, 
                        color=COLOR_ATTN, 
                        stroke_opacity=opacity
                    )
                    lines.add(l)
        
        self.play(Create(lines, lag_ratio=0.001), run_time=2.5)
        
        title = Text("Cross-Sectional Attention", font_size=32, color=COLOR_ATTN).to_corner(UL)
        sub = Text("Global Market Context (All-to-All)", font_size=24, color=WHITE).next_to(title, DOWN)
        self.add_fixed_in_frame_mobjects(title, sub)
        self.play(Write(title), FadeIn(sub))
        self.play(
            nodes.animate.set_color(WHITE).scale(1.1),
            lines.animate.set_stroke(color=WHITE, opacity=0.5),
            rate_func=there_and_back,
            run_time=1.5
        )
        self.wait(1)

        def get_cluster_indices(center_idx, count, exclude_idxs=[]):
            distances = []
            c_pos = node_positions[center_idx]
            for i, pos in enumerate(node_positions):
                if i in exclude_idxs: continue
                d = np.linalg.norm(c_pos - pos)
                distances.append((d, i))
            distances.sort()
            return [x[1] for x in distances[:count]]

        seed_1 = 0   
        seed_2 = 40
        seed_3 = 25 

        group1_idxs = get_cluster_indices(seed_1, 5, [])
        group2_idxs = get_cluster_indices(seed_2, 4, group1_idxs)
        group3_idxs = get_cluster_indices(seed_3, 3, group1_idxs + group2_idxs)

        heroes = {
            "Big Tech": {
                "idxs": group1_idxs, 
                "names": ["AAPL", "MSFT", "NVDA", "GOOGL", "META"], 
                "color": BLUE
            },
            "Finance": {
                "idxs": group2_idxs, 
                "names": ["JPM", "V", "MA", "BAC"], 
                "color": GREEN
            },
            "Pharma": {
                "idxs": group3_idxs, 
                "names": ["LLY", "JNJ", "PFE"], 
                "color": RED
            }
        }

        self.play(
            lines.animate.set_stroke(opacity=0.02), 
            nodes.animate.set_opacity(0.1),         
            run_time=1.0
        )

        hero_anims = []
        hero_labels = []
        attention_lines = VGroup()

        for sector, data in heroes.items():
            indices = data["idxs"]
            names = data["names"]
            color = data["color"]
            
            for i, idx in enumerate(indices):
                pos = node_positions[idx]
                
                dot = nodes[idx]
                hero_anims.append(dot.animate.set_color(color).set_opacity(1).scale(1.5))
                
                lbl_text = names[i]
                label = Text(lbl_text, font_size=24, color=color, weight=BOLD)
                
                label.rotate(90 * DEGREES, axis=RIGHT)
                label.rotate(target_theta - 90 * DEGREES, axis=UP)
                
                label.move_to(pos * 1.15)
                hero_labels.append(label)
                
                self.add_fixed_orientation_mobjects(label)

                for j in range(i + 1, len(indices)):
                    pos2 = node_positions[indices[j]]
                    line = Line(pos, pos2, color=color, stroke_width=3, stroke_opacity=0.8)
                    attention_lines.add(line)

        self.stop_ambient_camera_rotation()
        self.begin_ambient_camera_rotation(rate=0.02)
        
        self.play(
            *hero_anims, 
            Create(attention_lines),
            run_time=1.5
        )
        
        self.play(
            LaggedStart(*[FadeIn(l, scale=0.5) for l in hero_labels], lag_ratio=0.1),
            run_time=1.5
        )
        
        grp_nodes_list = [VGroup(*[nodes[i] for i in data["idxs"]]) for data in heroes.values()]
        grp_lines_list = []
        offset = 0
        for data in heroes.values():
            count = len(data["idxs"])
            grp_lines = VGroup(*attention_lines[offset:offset + (count * (count - 1)) // 2])
            grp_lines_list.append(grp_lines)
            offset += (count * (count - 1)) // 2

        self.play(
            *[Indicate(grp_nodes, color=WHITE, scale_factor=1.2) for grp_nodes in grp_nodes_list],
            *[Indicate(grp_lines, color=WHITE, scale_factor=1.02) for grp_lines in grp_lines_list],
            run_time=0.8
        )
        self.wait(2)

Manim Community v0.19.1